# Nama: Ahmad Fauzan  
# NIM: 23051030014  

# Support Vector Machines  

Author: Ahmad Fauzan  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmaddfauzan/Exercise-SVM/blob/main/Kode.ipynb)

# Exercise Part 1

# 1. Setup
Pada tahap ini dilakukan persiapan data untuk model SVM. 
Dataset MNIST digunakan sebagai data klasifikasi. 
Data kemudian dibagi menjadi data latih dan data uji.

SVM sangat sensitif terhadap skala fitur, sehingga digunakan StandardScaler 
dalam pipeline sebelum proses training.

## 1.1 Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC, LinearSVC, SVR, LinearSVR
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

import os

output_dir = "plots"
os.makedirs(output_dir, exist_ok=True)

## 1.2 Load Dataset (MNIST)

In [2]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1)
X, y = mnist["data"], mnist["target"]

# convert label ke integer
y = y.astype(np.int8)

## 1.3 Cek Dataset

In [3]:
print("Shape data:", X.shape)
print("Jumlah kelas:", np.unique(y))

Shape data: (70000, 784)
Jumlah kelas: [0 1 2 3 4 5 6 7 8 9]


## 1.4 Split Data (Train/Test)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 1.5 Pipeline + Scaling

In [5]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

# 2. Linear SVM Classification
Pada bagian ini dilakukan training model SVM linear untuk klasifikasi dataset MNIST. 
Tujuannya adalah mendapatkan baseline model sebelum dilakukan tuning atau penggunaan kernel non-linear.

Linear SVM bekerja dengan mencari hyperplane yang memaksimalkan margin antar kelas. 
Model ini cocok digunakan jika data dapat dipisahkan secara linear.

Parameter C digunakan untuk mengontrol trade-off antara margin yang besar 
dan jumlah kesalahan klasifikasi.

## 2.1 Training Linear SVM

In [ ]:
linear_svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1, loss="hinge", max_iter=10000))
])

linear_svm_pipeline.fit(X_train, y_train)

## 2.2 Prediksi

In [ ]:
y_pred = linear_svm_pipeline.predict(X_test)

## 2.3 Evaluasi Model

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9142142857142858

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.97      0.96      1343
           1       0.94      0.97      0.96      1600
           2       0.90      0.90      0.90      1380
           3       0.89      0.88      0.89      1433
           4       0.91      0.93      0.92      1295
           5       0.88      0.87      0.88      1273
           6       0.94      0.95      0.95      1396
           7       0.93      0.93      0.93      1503
           8       0.89      0.84      0.87      1357
           9       0.90      0.88      0.89      1420

    accuracy                           0.91     14000
   macro avg       0.91      0.91      0.91     14000
weighted avg       0.91      0.91      0.91     14000



# 3. Soft Margin Classification + Tuning Parameter C

## 3.1 Training dengan Beberapa Nilai C

In [ ]:
C_values = [0.1, 1, 10]

models = {}
accuracies = {}

for C in C_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(C=C, loss="hinge", max_iter=10000))
    ])
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    models[C] = model
    accuracies[C] = acc
    
    print(f"C = {C} → Accuracy = {acc}")

c:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


C = 0.1 → Accuracy = 0.9151428571428571


## 3.2 Membandingkan Hasil

In [ ]:
print("\nPerbandingan Accuracy:")
for C, acc in accuracies.items():
    print(f"C = {C}: {acc}")

In [ ]:
param_grid = {
    "svm__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(loss="hinge", max_iter=10000))
    ]),
    param_grid,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best C:", grid.best_params_)
print("Best Score:", grid.best_score_)

Dari hasil percobaan, terlihat bahwa nilai C mempengaruhi performa model.

Nilai C kecil menghasilkan margin yang lebih lebar namun toleransi terhadap error lebih tinggi, 
sehingga model cenderung lebih general namun kurang akurat.

Sebaliknya, nilai C besar membuat model lebih fokus mengurangi error pada data training, 
namun berisiko overfitting.

Pemilihan nilai C yang optimal dapat dilakukan menggunakan cross-validation.

# 4. NonLinear SVM Classification
Pada bagian ini digunakan SVM dengan kernel non-linear untuk menangani data 
yang tidak dapat dipisahkan secara linear.

Salah satu metode yang digunakan adalah kernel trick, yang memungkinkan data 
diproyeksikan ke dimensi yang lebih tinggi tanpa harus menghitungnya secara eksplisit.

Kernel yang umum digunakan adalah RBF (Radial Basis Function), yang memiliki 
parameter gamma (γ) untuk mengatur pengaruh setiap data terhadap decision boundary:
- gamma kecil → boundary lebih halus (smooth)
- gamma besar → boundary lebih kompleks

Tujuan bagian ini adalah membandingkan performa SVM linear dengan SVM non-linear.

## 4.1 Training SVM dengan RBF Kernel

In [ ]:
rbf_svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1, gamma="scale"))
])

rbf_svm_pipeline.fit(X_train, y_train)

## 4.2 Prediksi

In [ ]:
y_pred_rbf = rbf_svm_pipeline.predict(X_test)

## 4.3 Evaluasi

In [ ]:
print("Accuracy (RBF Kernel):", accuracy_score(y_test, y_pred_rbf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rbf))

In [ ]:
param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": [0.01, 0.1, "scale"]
}

grid_rbf = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf"))
    ]),
    param_grid,
    cv=3,
    n_jobs=-1
)

grid_rbf.fit(X_train, y_train)

print("Best Params:", grid_rbf.best_params_)
print("Best Score:", grid_rbf.best_score_)

SVM dengan kernel RBF mampu menangani data yang tidak dapat dipisahkan secara linear 
dengan cara memetakan data ke ruang berdimensi lebih tinggi.

Dibandingkan dengan Linear SVM, model ini biasanya menghasilkan akurasi yang lebih tinggi, 
namun dengan kompleksitas komputasi yang lebih besar.

Parameter gamma sangat berpengaruh terhadap bentuk decision boundary:
- gamma kecil → model lebih general
- gamma besar → model lebih kompleks dan berisiko overfitting

# 5. Polynomial Kernel
Pada bagian ini digunakan kernel polynomial untuk menangani hubungan non-linear 
antar data.

Kernel polynomial bekerja dengan menambahkan kombinasi fitur (feature expansion) 
sehingga data yang awalnya tidak linear dapat dipisahkan dalam ruang berdimensi lebih tinggi.

Parameter penting:
- degree → tingkat kompleksitas polinomial
- C → regularisasi
- coef0 → pengaruh terhadap high-degree vs low-degree

Tujuan bagian ini adalah membandingkan performa kernel polynomial dengan kernel RBF.

Kernel polynomial memungkinkan model menangkap hubungan non-linear antar fitur 
dengan meningkatkan dimensi fitur secara implisit.

Namun, dibandingkan dengan RBF kernel, polynomial kernel cenderung:
- lebih sensitif terhadap parameter degree
- kurang fleksibel untuk data kompleks seperti MNIST

Dalam banyak kasus, RBF kernel memberikan performa yang lebih baik dan lebih stabil.

## 5.1 Training Polynomial Kernel

In [ ]:
poly_svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="poly", degree=3, C=1, coef0=1))
])

poly_svm_pipeline.fit(X_train, y_train)

## 5.2 Prediksi

In [ ]:
y_pred_poly = poly_svm_pipeline.predict(X_test)

## 5.3 Evaluasi

In [ ]:
print("Accuracy (Polynomial Kernel):", accuracy_score(y_test, y_pred_poly))
print("\nClassification Report:\n", classification_report(y_test, y_pred_poly))

## 5.4 Tuning Polynomial Kernel

In [ ]:
param_grid_poly = {
    "svm__degree": [2, 3, 4],
    "svm__C": [0.1, 1, 10],
    "svm__coef0": [0, 1]
}

grid_poly = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="poly"))
    ]),
    param_grid_poly,
    cv=3,
    n_jobs=-1
)

grid_poly.fit(X_train, y_train)

print("Best Params (Poly):", grid_poly.best_params_)
print("Best Score (Poly):", grid_poly.best_score_)

## 5.5 Perbandingan Semua Model

In [ ]:
results = {
    "Linear SVM": accuracy_score(y_test, y_pred),
    "RBF SVM": accuracy_score(y_test, y_pred_rbf),
    "Polynomial SVM": accuracy_score(y_test, y_pred_poly)
}

for model, acc in results.items():
    print(f"{model}: {acc}")

In [ ]:
plt.bar(results.keys(), results.values())
plt.title("Perbandingan Akurasi Model SVM")
plt.ylabel("Accuracy")
plt.xticks(rotation=15)
plt.show()

Dari perbandingan model, terlihat bahwa:

- Linear SVM memberikan baseline yang cukup baik
- Polynomial kernel meningkatkan kemampuan model dalam menangkap pola non-linear
- RBF kernel umumnya memberikan performa terbaik karena fleksibilitasnya

Hal ini menunjukkan pentingnya pemilihan kernel dalam SVM untuk mendapatkan performa optimal.

# 6. SVM Regression
Pada bagian ini digunakan Support Vector Machine untuk tugas regresi (SVR). 
Berbeda dengan klasifikasi, SVM regression bertujuan untuk memprediksi nilai kontinu 
dengan mempertahankan error dalam batas toleransi tertentu (epsilon).

Parameter penting dalam SVR:
- epsilon → batas toleransi error
- C → regularisasi (trade-off antara error dan kompleksitas model)
- kernel → untuk menangani hubungan non-linear (misalnya RBF)

Dataset yang digunakan adalah California Housing untuk memprediksi harga rumah.

## 6.1 Load Dataset (California Housing)

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_reg, y_reg = housing.data, housing.target

## 6.2 Split Data

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

## 6.3 Linear SVR (Baseline)

In [ ]:
svr_linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", LinearSVR(epsilon=1.5, max_iter=10000))
])

svr_linear_pipeline.fit(X_train_reg, y_train_reg)

## 6.4 Prediksi & Evaluasi

In [ ]:
y_pred_reg_linear = svr_linear_pipeline.predict(X_test_reg)

rmse_linear = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_linear))

print("RMSE (Linear SVR):", rmse_linear)

## 6.5 SVR dengan RBF Kernel

In [ ]:
svr_rbf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=100, epsilon=0.1, gamma="scale"))
])

svr_rbf_pipeline.fit(X_train_reg, y_train_reg)

## 6.6 Evaluasi RBF SVR

In [ ]:
y_pred_reg_rbf = svr_rbf_pipeline.predict(X_test_reg)

rmse_rbf = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_rbf))

print("RMSE (RBF SVR):", rmse_rbf)

## 6.7 Hyperparameter Tuning

In [ ]:
from scipy.stats import reciprocal, uniform
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "svr__C": reciprocal(1, 100),
    "svr__gamma": reciprocal(0.01, 1),
    "svr__epsilon": uniform(0.01, 0.2)
}

random_search = RandomizedSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf"))
    ]),
    param_distributions,
    n_iter=20,
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_reg, y_train_reg)

print("Best Params:", random_search.best_params_)

## 6.8 Evaluasi Model Terbaik

In [ ]:
best_model = random_search.best_estimator_

y_pred_best = best_model.predict(X_test_reg)

rmse_best = np.sqrt(mean_squared_error(y_test_reg, y_pred_best))

print("RMSE (Best Model):", rmse_best)

## 6.9 Perbandingan Model

In [ ]:
print("Perbandingan RMSE:")
print(f"Linear SVR: {rmse_linear}")
print(f"RBF SVR: {rmse_rbf}")
print(f"Best Tuned SVR: {rmse_best}")

Hasil menunjukkan bahwa SVR dengan kernel RBF memberikan performa yang lebih baik 
dibandingkan Linear SVR karena mampu menangkap hubungan non-linear dalam data.

Hyperparameter tuning menggunakan RandomizedSearchCV membantu menemukan kombinasi 
parameter terbaik sehingga meningkatkan performa model.

Nilai RMSE yang lebih kecil menunjukkan model yang lebih baik dalam memprediksi harga rumah.

# 7. Comparison: Linear SVC vs SVC vs SGD Classifier
Pada bagian ini dilakukan perbandingan beberapa model klasifikasi linear, yaitu:
- LinearSVC
- SVC dengan kernel linear
- SGDClassifier

Tujuan dari perbandingan ini adalah untuk memahami perbedaan performa, 
kompleksitas, dan karakteristik masing-masing model.

LinearSVC dan SVC sama-sama berbasis SVM, namun menggunakan pendekatan optimisasi yang berbeda.
SGDClassifier menggunakan pendekatan stochastic gradient descent yang lebih cepat untuk dataset besar.

## 7.1 Linear SVC

In [ ]:
linear_svc_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(C=1, max_iter=10000))
])

linear_svc_model.fit(X_train, y_train)

y_pred_linear_svc = linear_svc_model.predict(X_test)

acc_linear_svc = accuracy_score(y_test, y_pred_linear_svc)

print("LinearSVC Accuracy:", acc_linear_svc)

## 7.2 SVC dengan Kernel Linear

In [ ]:
svc_linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="linear", C=1))
])

svc_linear_model.fit(X_train, y_train)

y_pred_svc_linear = svc_linear_model.predict(X_test)

acc_svc_linear = accuracy_score(y_test, y_pred_svc_linear)

print("SVC (Linear Kernel) Accuracy:", acc_svc_linear)

## 7.3 SGD Classifier

In [ ]:
sgd_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SGDClassifier(loss="hinge", max_iter=1000, tol=1e-3))
])

sgd_model.fit(X_train, y_train)

y_pred_sgd = sgd_model.predict(X_test)

acc_sgd = accuracy_score(y_test, y_pred_sgd)

print("SGDClassifier Accuracy:", acc_sgd)

## 7.4 Perbandingan Hasil

In [ ]:
comparison_results = {
    "LinearSVC": acc_linear_svc,
    "SVC (Linear)": acc_svc_linear,
    "SGDClassifier": acc_sgd
}

for model, acc in comparison_results.items():
    print(f"{model}: {acc}")

In [ ]:
plt.bar(comparison_results.keys(), comparison_results.values())
plt.title("Perbandingan Model Linear Classifier")
plt.ylabel("Accuracy")
plt.xticks(rotation=15)
plt.show()

In [ ]:
import time

models_time = {}

for name, model in [
    ("LinearSVC", linear_svc_model),
    ("SVC Linear", svc_linear_model),
    ("SGDClassifier", sgd_model)
]:
    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()
    
    models_time[name] = end - start

print("\nTraining Time:")
for model, t in models_time.items():
    print(f"{model}: {t:.4f} detik")

Dari hasil perbandingan:

- LinearSVC:
  Cepat dan efisien untuk dataset besar, dengan performa yang cukup baik.

- SVC dengan kernel linear:
  Lebih akurat dalam beberapa kasus, namun lebih lambat karena menggunakan metode optimisasi yang lebih kompleks.

- SGDClassifier:
  Sangat cepat dan scalable untuk dataset besar, namun akurasi biasanya sedikit di bawah SVM.

Kesimpulannya, pemilihan model tergantung pada kebutuhan:
- Akurasi tinggi → SVC
- Kecepatan → SGDClassifier
- Balance → LinearSVC

# Kesimpulan

SVM merupakan model yang sangat powerful untuk klasifikasi dan regresi. 
Pemilihan kernel dan parameter seperti C dan gamma sangat mempengaruhi performa model.

Kernel RBF terbukti memberikan performa terbaik untuk data non-linear, 
sedangkan LinearSVC dan SGDClassifier cocok untuk dataset besar dengan kebutuhan komputasi cepat.


# Exercise Part 2

# 8. SVM Classification on Titanic Dataset
Pada bagian ini dilakukan klasifikasi menggunakan SVM pada dataset Titanic 
untuk memprediksi apakah penumpang selamat atau tidak.

Dataset ini memiliki data kategorikal dan missing values, sehingga diperlukan 
preprocessing sebelum dilakukan training model.

## 8.1 Load Dataset

In [ ]:
import seaborn as sns

titanic = sns.load_dataset("titanic")
titanic.head()

## 8.2 Preprocessiong

In [ ]:
data = titanic[["survived", "pclass", "sex", "age", "fare", "embarked"]]

# handle missing values
data["age"].fillna(data["age"].median(), inplace=True)
data["fare"].fillna(data["fare"].median(), inplace=True)
data["embarked"].fillna(data["embarked"].mode()[0], inplace=True)

# encoding
data["sex"] = data["sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["embarked"], drop_first=True)

## 8.3 Split Data

In [ ]:
X_titanic = data.drop("survived", axis=1)
y_titanic = data["survived"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42
)

## 8.4 Baseline Model (RBF SVM)

In [ ]:
svm_titanic = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1, gamma="scale"))
])

svm_titanic.fit(X_train_t, y_train_t)

y_pred_t = svm_titanic.predict(X_test_t)
acc_before = accuracy_score(y_test_t, y_pred_t)

print("Accuracy (Baseline):", acc_before)

## 8.5 Tuning

In [ ]:
param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": [0.01, 0.1, "scale"]
}

grid = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf"))
    ]),
    param_grid,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_t, y_train_t)

print("Best Params:", grid.best_params_)

## 8.6 Evaluasi Model Terbaik

In [ ]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test_t)
acc_after = accuracy_score(y_test_t, y_pred_best)

print("Accuracy (Best):", acc_after)
print("\nClassification Report:\n", classification_report(y_test_t, y_pred_best))

In [ ]:
plt.figure()
plt.bar(["Baseline", "Tuned"], [acc_before, acc_after])
plt.title("Titanic SVM Accuracy Comparison")
plt.ylabel("Accuracy")

plt.savefig(os.path.join(output_dir, "titanic_accuracy.png"), dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(best_model, X_test_t, y_test_t)

plt.title("Titanic Confusion Matrix")
plt.savefig(os.path.join(output_dir, "titanic_confusion_matrix.png"), dpi=300)
plt.show()

# 9. Effect of C & Gamma (RBF Kernel)
Bagian ini mengeksplorasi pengaruh parameter C dan gamma terhadap performa model SVM.

## 9.1 Eksperimen Manual

In [ ]:
C_values = [0.1, 1, 10]
gamma_values = [0.01, 0.1, 1]

results = []

for C in C_values:
    for gamma in gamma_values:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="rbf", C=C, gamma=gamma))
        ])
        
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        
        results.append((C, gamma, acc))
        print(f"C={C}, gamma={gamma} → {acc}")

## 9.2 heatmap

In [ ]:
df = pd.DataFrame(results, columns=["C", "gamma", "accuracy"])
pivot = df.pivot(index="C", columns="gamma", values="accuracy")

plt.figure()
plt.imshow(pivot, aspect='auto')
plt.colorbar()

plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)

plt.xlabel("gamma")
plt.ylabel("C")
plt.title("Effect of C and Gamma")

plt.savefig(os.path.join(output_dir, "rbf_heatmap.png"), dpi=300)
plt.show()

# 10. One-Class SVM (Anomaly Detection)

## 10.1 Training

In [ ]:
from sklearn.svm import OneClassSVM

X_normal = X_train[y_train == 0]

one_class = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", OneClassSVM(kernel="rbf", gamma=0.1, nu=0.1))
])

one_class.fit(X_normal)

## 10.2 Deteksi

In [ ]:
pred = one_class.predict(X_test[:200])

## 10.3 Visualisasi

In [ ]:
plt.figure()
plt.hist(pred, bins=3)
plt.title("Anomaly Detection Result (-1=Outlier, 1=Normal)")

plt.savefig(os.path.join(output_dir, "anomaly_hist.png"), dpi=300)
plt.show()